In [ ]:
import pandas as pd

# =============================================
# 데이터 불러오기
# → 파일이 data/raw/ 폴더 안에 있음!
# =============================================

orders   = pd.read_csv('../data/raw/orders.csv')
prior_products = pd.read_csv('../data/raw/order_products__prior.csv')

print('orders 크기:', orders.shape)
print('prior_products 크기:', prior_products.shape)
print(orders.head())


# =============================================
# 메모리 최적화
# (데이터가 너무 크기 때문에 꼭 필요!)
# =============================================

before = orders.memory_usage(deep=True).sum() / 1024**2
print('최적화 전 메모리:', round(before, 1), 'MB')

orders['order_dow']              = orders['order_dow'].astype('int8')
orders['order_hour_of_day']      = orders['order_hour_of_day'].astype('int8')
orders['order_number']           = orders['order_number'].astype('int16')
orders['days_since_prior_order'] = orders['days_since_prior_order'].astype('float32')

prior_products['reordered']  = prior_products['reordered'].astype('int8')
prior_products['product_id'] = prior_products['product_id'].astype('int32')

after = orders.memory_usage(deep=True).sum() / 1024**2
print('최적화 후 메모리:', round(after, 1), 'MB')


# =============================================
# prior 데이터만 사용
# → prior = 유저의 과거 구매 기록 (feature 만드는 재료)
# → train = 정답(Label) 용도라 feature 만들 때 쓰면 안 됨
# =============================================

prior_orders = orders[orders['eval_set'] == 'prior']

print('prior 주문 수:', len(prior_orders))


# =============================================
# Feature 1 : user_total_orders (총 주문 횟수)
# → order_number 최댓값 = 유저가 지금까지 몇 번 주문했는지
# =============================================

user_total_orders = prior_orders.groupby('user_id')['order_number'].max()
user_total_orders = user_total_orders.reset_index()
user_total_orders.columns = ['user_id', 'user_total_orders']

print('user_total_orders:')
print(user_total_orders.head())


# =============================================
# Feature 2 : user_avg_days_between (평균 재구매 주기)
# → EDA에서 평균 11.1일, 7일/30일 주기 패턴을 확인했음
# → 첫 주문은 NaN 이니까 제외하고 계산
# =============================================

재구매데이터 = prior_orders[prior_orders['days_since_prior_order'].notnull()]

user_avg_days = 재구매데이터.groupby('user_id')['days_since_prior_order'].mean()
user_avg_days = user_avg_days.reset_index()
user_avg_days.columns = ['user_id', 'user_avg_days_between']

print('user_avg_days_between:')
print(user_avg_days.head())


# =============================================
# Feature 3 : user_std_days_between (구매 주기 규칙성)
# → 표준편차가 낮을수록 규칙적으로 구매하는 유저
# → 규칙적인 유저일수록 다음 주문 예측이 쉬움
# =============================================

user_std_days = 재구매데이터.groupby('user_id')['days_since_prior_order'].std()
user_std_days = user_std_days.reset_index()
user_std_days.columns = ['user_id', 'user_std_days_between']

print('user_std_days_between:')
print(user_std_days.head())


# =============================================
# prior_products 와 prior_orders 합치기
# → product_id, reordered 정보를 가져오려면 order_id 기준으로 합쳐야 함
# =============================================

prior_merged = prior_products.merge(prior_orders[['order_id', 'user_id']], on='order_id')

print('합친 후 크기:', prior_merged.shape)


# =============================================
# Feature 4 : user_total_items (총 구매 상품 수)
# → 유저가 지금까지 산 상품 개수 합계
# =============================================

user_total_items = prior_merged.groupby('user_id')['product_id'].count()
user_total_items = user_total_items.reset_index()
user_total_items.columns = ['user_id', 'user_total_items']

print('user_total_items:')
print(user_total_items.head())


# =============================================
# Feature 5 : user_distinct_items (경험해본 서로 다른 상품 수)
# → nunique = 중복 제거하고 몇 가지 상품을 샀는지
# =============================================

user_distinct_items = prior_merged.groupby('user_id')['product_id'].nunique()
user_distinct_items = user_distinct_items.reset_index()
user_distinct_items.columns = ['user_id', 'user_distinct_items']

print('user_distinct_items:')
print(user_distinct_items.head())


# =============================================
# Feature 6 : user_reorder_ratio (재구매 성향)
# → 전체 구매 중 reordered=1 인 비율
# → 높을수록 같은 상품을 반복해서 사는 유저
# =============================================

user_reorder_ratio = prior_merged.groupby('user_id')['reordered'].mean()
user_reorder_ratio = user_reorder_ratio.reset_index()
user_reorder_ratio.columns = ['user_id', 'user_reorder_ratio']

print('user_reorder_ratio:')
print(user_reorder_ratio.head())


# =============================================
# Feature 7 : favorite_dow (선호 요일 최빈값)
# → EDA에서 일요일/월요일에 주문 집중 패턴 확인했음
# =============================================

favorite_dow = prior_orders.groupby('user_id')['order_dow'].agg(lambda x: x.mode()[0])
favorite_dow = favorite_dow.reset_index()
favorite_dow.columns = ['user_id', 'favorite_dow']

print('favorite_dow:')
print(favorite_dow.head())


# =============================================
# Feature 8 : favorite_hour (선호 시간대 최빈값)
# → EDA에서 오전 10시 피크 패턴 확인했음
# =============================================

favorite_hour = prior_orders.groupby('user_id')['order_hour_of_day'].agg(lambda x: x.mode()[0])
favorite_hour = favorite_hour.reset_index()
favorite_hour.columns = ['user_id', 'favorite_hour']

print('favorite_hour:')
print(favorite_hour.head())


# =============================================
# 전부 합치기 (user_id 기준으로 하나씩 붙이기)
# =============================================

user_features = user_total_orders.merge(user_avg_days,       on='user_id')
user_features = user_features.merge(user_std_days,           on='user_id')
user_features = user_features.merge(user_total_items,        on='user_id')
user_features = user_features.merge(user_distinct_items,     on='user_id')
user_features = user_features.merge(user_reorder_ratio,      on='user_id')
user_features = user_features.merge(favorite_dow,            on='user_id')
user_features = user_features.merge(favorite_hour,           on='user_id')

print('최종 user_features:')
print(user_features.head(10))
print('크기:', user_features.shape)


# =============================================
# 합친 후 메모리 최적화
# =============================================

user_features['user_total_orders']    = user_features['user_total_orders'].astype('int16')
user_features['user_avg_days_between']= user_features['user_avg_days_between'].astype('float32')
user_features['user_std_days_between']= user_features['user_std_days_between'].astype('float32')
user_features['user_total_items']     = user_features['user_total_items'].astype('int32')
user_features['user_distinct_items']  = user_features['user_distinct_items'].astype('int16')
user_features['user_reorder_ratio']   = user_features['user_reorder_ratio'].astype('float32')
user_features['favorite_dow']         = user_features['favorite_dow'].astype('int8')
user_features['favorite_hour']        = user_features['favorite_hour'].astype('int8')

print('최종 메모리:', round(user_features.memory_usage(deep=True).sum() / 1024**2, 1), 'MB')


# =============================================
# CSV 저장
# =============================================

user_features.to_csv('../data/prep/user_features.csv', index=False)
print('저장 완료! → data/prep/user_features.csv')
